# Your First RAG Application

In this notebook, we'll walk you through each of the components that are involved in a simple RAG application.

We won't be leveraging any fancy tools, just the OpenAI Python SDK, Numpy, and some classic Python.

> NOTE: This was done with Python 3.11.4.

> NOTE: There might be [compatibility issues](https://github.com/wandb/wandb/issues/7683) if you're on NVIDIA driver >552.44 As an interim solution - you can rollback your drivers to the 552.44.

## Table of Contents:

- Task 1: Imports and Utilities
- Task 2: Documents
- Task 3: Embeddings and Vectors
- Task 4: Prompts
- Task 5: Retrieval Augmented Generation
  - 🚧 Activity #1: Augment RAG

Let's look at a rather complicated looking visual representation of a basic RAG application.

<img src="https://i.imgur.com/vD8b016.png" />

## Task 1: Imports and Utility

We're just doing some imports and enabling `async` to work within the Jupyter environment here, nothing too crazy!

In [24]:
from aimakerspace.text_utils import TextFileLoader, CharacterTextSplitter
from aimakerspace.vectordatabase import VectorDatabase
import asyncio

In [25]:
import nest_asyncio
nest_asyncio.apply()

## Task 2: Documents

We'll be concerning ourselves with this part of the flow in the following section:

<img src="https://i.imgur.com/jTm9gjk.png" />

### Loading Source Documents

So, first things first, we need some documents to work with.

While we could work directly with the `.txt` files (or whatever file-types you wanted to extend this to) we can instead do some batch processing of those documents at the beginning in order to store them in a more machine compatible format.

In this case, we're going to parse our text file into a single document in memory.

Let's look at the relevant bits of the `TextFileLoader` class:

```python
def load_file(self):
        with open(self.path, "r", encoding=self.encoding) as f:
            self.documents.append(f.read())
```

We're simply loading the document using the built in `open` method, and storing that output in our `self.documents` list.

> NOTE: We're using blogs from PMarca (Marc Andreessen) as our sample data. This data is largely irrelevant as we want to focus on the mechanisms of RAG, which includes out data's shape and quality - but not specifically what the contents of the data are. 


In [26]:
from tkinter import filedialog
text_loader = TextFileLoader(filedialog.askopenfilename(title="Select PDF file", filetypes=[("PDF files", "*.pdf")]))
documents = text_loader.load_documents()
len(documents)

1

In [27]:
print(documents[0][:100])

ARTIFICIAL  
INTELLIGENCE 
STRATEGIC PLAN
Fiscal Years 2023-2027U.S. NUCLEAR REGULATORY COMMISSIONNU


### Splitting Text Into Chunks

As we can see, there is one massive document.

We'll want to chunk the document into smaller parts so it's easier to pass the most relevant snippets to the LLM.

There is no fixed way to split/chunk documents - and you'll need to rely on some intuition as well as knowing your data *very* well in order to build the most robust system.

For this toy example, we'll just split blindly on length.

>There's an opportunity to clear up some terminology here, for this course we will be stick to the following:
>
>- "source documents" : The `.txt`, `.pdf`, `.html`, ..., files that make up the files and information we start with in its raw format
>- "document(s)" : single (or more) text object(s)
>- "corpus" : the combination of all of our documents

As you can imagine (though it's not specifically true in this toy example) the idea of splitting documents is to break them into managable sized chunks that retain the most relevant local context.

In [28]:
text_splitter = CharacterTextSplitter()
split_documents = text_splitter.split_texts(documents)
len(split_documents)

71

Let's take a look at some of the documents we've managed to split.

In [29]:
split_documents[0:1]

['ARTIFICIAL  \nINTELLIGENCE \nSTRATEGIC PLAN\nFiscal Years 2023-2027U.S. NUCLEAR REGULATORY COMMISSIONNUREG-2261AVAILABILITY OF REFERENCE MATERIALS\nIN NRC PUBLICATIONS\nNRC Reference Material\nAs of November 1999, you may electronically access \nNUREG-series publications and other NRC records at  the \nNRC’s Library at www.nrc.gov/reading-rm.html. Publicly \nreleased records include, to name a few, NUREG-series \npublications; Federal Register  notices; applicant, licensee, \nand vendor documents and correspondence; NRC \ncorrespondence and internal memoranda; bulletins and \ninformation notices; inspection and investigative reports; \nlicensee event reports; and Commission papers and their \nattachments.\nNRC publications in the NUREG series, NRC regulations, \nand Title 10, “Energy,” in the Code of Federal Regulations  \nmay also be purchased from one of these two sources :\n1. The Superintendent of Documents\nU.S. Government Publishing Office\nWashington,  DC  20402-0001\nInternet

## Task 3: Embeddings and Vectors

Next, we have to convert our corpus into a "machine readable" format as we explored in the Embedding Primer notebook.

Today, we're going to talk about the actual process of creating, and then storing, these embeddings, and how we can leverage that to intelligently add context to our queries.

### OpenAI API Key

In order to access OpenAI's APIs, we'll need to provide our OpenAI API Key!

You can work through the folder "OpenAI API Key Setup" for more information on this process if you don't already have an API Key!

In [30]:
import os
import openai
from getpass import getpass
from dotenv import load_dotenv
from pathlib import Path

# Get the current directory
current_dir = Path.cwd()
print(f"Current directory: {current_dir}")
print(f".env file exists: {(current_dir / '.env').exists()}")

# Load environment variables
load_dotenv(override=True)

# Print the API key (first few characters only for security)
api_key = os.environ.get("OPENAI_API_KEY")
if api_key:
    print(f"API Key loaded: {api_key[:10]}...")
else:
    print("No API key found in environment")

# Set up OpenAI API key
if api_key and len(api_key) > 0:
    openai.api_key = api_key
else:
    openai.api_key = getpass("OpenAI API Key: ")
    os.environ["OPENAI_API_KEY"] = openai.api_key


Current directory: /home/mbudisic/Documents/AIE6/02_Embeddings_and_RAG
.env file exists: True
API Key loaded: sk-proj-Hn...


### Vector Database

Let's set up our vector database to hold all our documents and their embeddings!

While this is all baked into 1 call - we can look at some of the code that powers this process to get a better understanding:

Let's look at our `VectorDatabase().__init__()`:

```python
def __init__(self, embedding_model: EmbeddingModel = None):
        self.vectors = defaultdict(np.array)
        self.embedding_model = embedding_model or EmbeddingModel()
```

As you can see - our vectors are merely stored as a dictionary of `np.array` objects.

Secondly, our `VectorDatabase()` has a default `EmbeddingModel()` which is a wrapper for OpenAI's `text-embedding-3-small` model.

> **Quick Info About `text-embedding-3-small`**:
> - It has a context window of **8191** tokens
> - It returns vectors with dimension **1536**

#### ❓Question #1:

The default embedding dimension of `text-embedding-3-small` is 1536, as noted above. 

1. Is there any way to modify this dimension?
2. What technique does OpenAI use to achieve this?

> NOTE: Check out this [API documentation](https://platform.openai.com/docs/api-reference/embeddings/create) for the answer to question #1, and [this documentation](https://platform.openai.com/docs/guides/embeddings/use-cases) for an answer to question #2!

#### ❗Answer #1:

1. Yes, the dimension can be reduced through the use of OpenAI's API.
2. OpenAI uses so-called "Matryoshka" (slavic nested dolls) approach by training the embedding model to produce multiple nested representation spaces. 
    E.g. space of dimension 8 and dimension 16 are related to each other in the way that the first 8 coordinates in the 16-dim representation are the same as those in the 8-dim representation.

We can call the `async_get_embeddings` method of our `EmbeddingModel()` on a list of `str` and receive a list of `float` back!

```python
async def async_get_embeddings(self, list_of_text: List[str]) -> List[List[float]]:
        return await aget_embeddings(
            list_of_text=list_of_text, engine=self.embeddings_model_name
        )
```

We cast those to `np.array` when we build our `VectorDatabase()`:

```python
async def abuild_from_list(self, list_of_text: List[str]) -> "VectorDatabase":
        embeddings = await self.embedding_model.async_get_embeddings(list_of_text)
        for text, embedding in zip(list_of_text, embeddings):
            self.insert(text, np.array(embedding))
        return self
```

And that's all we need to do!

In [31]:
vector_db = VectorDatabase()
vector_db = asyncio.run(vector_db.abuild_from_list(split_documents))

#### ❓Question #2:

What are the benefits of using an `async` approach to collecting our embeddings?


> NOTE: Determining the core difference between `async` and `sync` will be useful! If you get stuck - ask ChatGPT!

#### ❗Answer #2:

Async calls allow us to process many batches in parallel, thus saving time and improving scalability if there is enough processing cores/threads.

It also keeps the UI responsive (presumably all of those async threads are distinct from the UI thread).


So, to review what we've done so far in natural language:

1. We load source documents
2. We split those source documents into smaller chunks (documents)
3. We send each of those documents to the `text-embedding-3-small` OpenAI API endpoint
4. We store each of the text representations with the vector representations as keys/values in a dictionary

### Semantic Similarity

The next step is to be able to query our `VectorDatabase()` with a `str` and have it return to us vectors and text that is most relevant from our corpus.

We're going to use the following process to achieve this in our toy example:

1. We need to embed our query with the same `EmbeddingModel()` as we used to construct our `VectorDatabase()`
2. We loop through every vector in our `VectorDatabase()` and use a distance measure to compare how related they are
3. We return a list of the top `k` closest vectors, with their text representations

There's some very heavy optimization that can be done at each of these steps - but let's just focus on the basic pattern in this notebook.

> We are using [cosine similarity](https://www.engati.com/glossary/cosine-similarity) as a distance metric in this example - but there are many many distance metrics you could use - like [these](https://flavien-vidal.medium.com/similarity-distances-for-natural-language-processing-16f63cd5ba55)

> We are using a rather inefficient way of calculating relative distance between the query vector and all other vectors - there are more advanced approaches that are much more efficient, like [ANN](https://towardsdatascience.com/comprehensive-guide-to-approximate-nearest-neighbors-algorithms-8b94f057d6b6)

In [32]:
vector_db.search_by_text("What are the main goals NRC is trying to accomplish through use of AI?", k=3)

[(' to contribute to AI research and development activities.   \nThe NRC will cultivate the talent of its existing highly skilled workforce by investing in \ncomprehensive training for NRC staff and managers working on use cases (AI Strategic \nGoal  5). The NRC AI training program will use a tiered approach, providing training \nranging from basic to advanced concepts, applications, and AI tools tailored to the needs of the agency and staff development objectives.  \nThe goal is to adopt the appropriate training programs and tools to develop the \nrequisite skills in the NRC workforce. A successful outcome of this goal is to ensure appropriate qualifications, training, expertise, and access to tools exist for the workforce to review and evaluate AI usage in NRC-regulated activities effectively, efficiently and in a timely manner.\n4.5  Str ategic Goal 5: Pursue Use Cases to Build an AI Foundation \nAcross the NRC\nAI technologies may pose novel challenges for the NRC regulatory framew

## Task 4: Prompts

In the following section, we'll be looking at the role of prompts - and how they help us to guide our application in the right direction.

In this notebook, we're going to rely on the idea of "zero-shot in-context learning".

This is a lot of words to say: "We will ask it to perform our desired task in the prompt, and provide no examples."

### XYZRolePrompt

Before we do that, let's stop and think a bit about how OpenAI's chat models work.

We know they have roles - as is indicated in the following API [documentation](https://platform.openai.com/docs/api-reference/chat/create#chat/create-messages)

There are three roles, and they function as follows (taken directly from [OpenAI](https://platform.openai.com/docs/guides/gpt/chat-completions-api)):

- `{"role" : "system"}` : The system message helps set the behavior of the assistant. For example, you can modify the personality of the assistant or provide specific instructions about how it should behave throughout the conversation. However note that the system message is optional and the model’s behavior without a system message is likely to be similar to using a generic message such as "You are a helpful assistant."
- `{"role" : "user"}` : The user messages provide requests or comments for the assistant to respond to.
- `{"role" : "assistant"}` : Assistant messages store previous assistant responses, but can also be written by you to give examples of desired behavior.

The main idea is this:

1. You start with a system message that outlines how the LLM should respond, what kind of behaviours you can expect from it, and more
2. Then, you can provide a few examples in the form of "assistant"/"user" pairs
3. Then, you prompt the model with the true "user" message.

In this example, we'll be forgoing the 2nd step for simplicities sake.

#### Utility Functions

You'll notice that we're using some utility functions from the `aimakerspace` module - let's take a peek at these and see what they're doing!

##### XYZRolePrompt

Here we have our `system`, `user`, and `assistant` role prompts.

Let's take a peek at what they look like:

```python
class BasePrompt:
    def __init__(self, prompt):
        """
        Initializes the BasePrompt object with a prompt template.

        :param prompt: A string that can contain placeholders within curly braces
        """
        self.prompt = prompt
        self._pattern = re.compile(r"\{([^}]+)\}")

    def format_prompt(self, **kwargs):
        """
        Formats the prompt string using the keyword arguments provided.

        :param kwargs: The values to substitute into the prompt string
        :return: The formatted prompt string
        """
        matches = self._pattern.findall(self.prompt)
        return self.prompt.format(**{match: kwargs.get(match, "") for match in matches})

    def get_input_variables(self):
        """
        Gets the list of input variable names from the prompt string.

        :return: List of input variable names
        """
        return self._pattern.findall(self.prompt)
```

Then we have our `RolePrompt` which laser focuses us on the role pattern found in most API endpoints for LLMs.

```python
class RolePrompt(BasePrompt):
    def __init__(self, prompt, role: str):
        """
        Initializes the RolePrompt object with a prompt template and a role.

        :param prompt: A string that can contain placeholders within curly braces
        :param role: The role for the message ('system', 'user', or 'assistant')
        """
        super().__init__(prompt)
        self.role = role

    def create_message(self, **kwargs):
        """
        Creates a message dictionary with a role and a formatted message.

        :param kwargs: The values to substitute into the prompt string
        :return: Dictionary containing the role and the formatted message
        """
        return {"role": self.role, "content": self.format_prompt(**kwargs)}
```

We'll look at how the `SystemRolePrompt` is constructed to get a better idea of how that extension works:

```python
class SystemRolePrompt(RolePrompt):
    def __init__(self, prompt: str):
        super().__init__(prompt, "system")
```

That pattern is repeated for our `UserRolePrompt` and our `AssistantRolePrompt` as well.

##### ChatOpenAI

Next we have our model, which is converted to a format analagous to libraries like LangChain and LlamaIndex.

Let's take a peek at how that is constructed:

```python
class ChatOpenAI:
    def __init__(self, model_name: str = "gpt-4o-mini"):
        self.model_name = model_name
        self.openai_api_key = os.getenv("OPENAI_API_KEY")
        if self.openai_api_key is None:
            raise ValueError("OPENAI_API_KEY is not set")

    def run(self, messages, text_only: bool = True):
        if not isinstance(messages, list):
            raise ValueError("messages must be a list")

        openai.api_key = self.openai_api_key
        response = openai.ChatCompletion.create(
            model=self.model_name, messages=messages
        )

        if text_only:
            return response.choices[0].message.content

        return response
```

#### ❓ Question #3:

When calling the OpenAI API - are there any ways we can achieve more reproducible outputs?

> NOTE: Check out [this section](https://platform.openai.com/docs/guides/text-generation/) of the OpenAI documentation for the answer!

#### ❗Answer #3:

There are several ways - the most straightforward is to reduce the temperature to zero.
In this way the model will always sample the most likely next token, rather than
randomly sampling tokens with a probability correlated to their likelihood.
Top-P can also help if temperature is not set to zero - it defines the cutoff for the tail
end of the token distribution that will be discarded before sampling. Smaller top-P retains
fewer tokens (which are then sampled from when temp > 0).

Prompt engineering is still important - asking the LLM to stick to the instructions/context,
to state "I don't know" instead of taking a wild guess, and asking it to be literal instead
of creative may all help.





### Creating and Prompting OpenAI's `gpt-4o-mini`!

Let's tie all these together and use it to prompt `gpt-4o-mini`!

In [33]:
from aimakerspace.openai_utils.prompts import (
    UserRolePrompt,
    SystemRolePrompt,
    AssistantRolePrompt,
)

from aimakerspace.openai_utils.chatmodel import ChatOpenAI

chat_openai = ChatOpenAI()
user_prompt_template = "{content}"
user_role_prompt = UserRolePrompt(user_prompt_template)
system_prompt_template = (
    "You are an expert in {expertise}, you always answer in a kind way."
)
system_role_prompt = SystemRolePrompt(system_prompt_template)

messages = [
    system_role_prompt.create_message(expertise="Python"),
    user_role_prompt.create_message(
        content="What is the best way to write a loop?"
    ),
]

response = chat_openai.run(messages)

In [34]:
print(response)

The best way to write a loop in Python often depends on the specific use case you have in mind. Below are a few common types of loops and some best practices for each:

### 1. **For Loop**
For iterating over a sequence (like a list, tuple, or string), a `for` loop is usually the best choice.

```python
# Example: Iterating over a list of numbers
numbers = [1, 2, 3, 4, 5]
for number in numbers:
    print(number)
```

**Best Practices:**
- Use meaningful variable names.
- Keep the loop body concise.
- Consider using list comprehensions for simple transformations.

### 2. **While Loop**
Use a `while` loop when the number of iterations isn't known in advance and depends on a condition.

```python
# Example: Counting down from 5
count = 5
while count > 0:
    print(count)
    count -= 1
```

**Best Practices:**
- Ensure the loop has a termination condition to avoid infinite loops.
- Be cautious with complex conditions.

### 3. **Using `enumerate` for Index and Value**
When you need both the

## Task 5: Retrieval Augmented Generation

Now we can create a RAG prompt - which will help our system behave in a way that makes sense!

There is much you could do here, many tweaks and improvements to be made!

In [35]:
RAG_PROMPT_TEMPLATE = """ \
Use the provided context to answer the user's query.

You may not answer the user's query unless there is specific context in the following text.

If you do not know the answer, or cannot answer, please respond with "I don't know".
"""

rag_prompt = SystemRolePrompt(RAG_PROMPT_TEMPLATE)

USER_PROMPT_TEMPLATE = """ \
Context:
{context}

User Query:
{user_query}
"""


user_prompt = UserRolePrompt(USER_PROMPT_TEMPLATE)

class RetrievalAugmentedQAPipeline:
    def __init__(self, llm: ChatOpenAI(), vector_db_retriever: VectorDatabase) -> None:
        self.llm = llm
        self.vector_db_retriever = vector_db_retriever

    def run_pipeline(self, user_query: str) -> str:
        context_list = self.vector_db_retriever.search_by_text(user_query, k=4)

        context_prompt = ""
        for context in context_list:
            context_prompt += context[0] + "\n"

        formatted_system_prompt = rag_prompt.create_message()

        formatted_user_prompt = user_prompt.create_message(user_query=user_query, context=context_prompt)

        return {"user_query" : user_query, "response" : self.llm.run([formatted_system_prompt, formatted_user_prompt]), "context" : context_list}

#### ❓ Question #4:

What prompting strategies could you use to make the LLM have a more thoughtful, detailed response?

What is that strategy called?

> NOTE: You can look through ["Accessing GPT-3.5-turbo Like a Developer"](https://colab.research.google.com/drive/1mOzbgf4a2SP5qQj33ZxTz2a01-5eXqk2?usp=sharing) for an answer to this question if you get stuck!

#### ❗Answer #4:

We can use prompt engineering [best practices](https://arxiv.org/pdf/2312.16171):

- threaten model with penalties or entice with a tip
- if used to summarize, provide a few-shot example, especially if there is an 
    internal pattern in documents that is repeated,
- ask for explanation in simple terms, or ask it to generate a list.




In [36]:
retrieval_augmented_qa_pipeline = RetrievalAugmentedQAPipeline(
    vector_db_retriever=vector_db,
    llm=chat_openai
)

In [37]:
answer = retrieval_augmented_qa_pipeline.run_pipeline("What are the main goals NRC is trying to accomplish through use of AI?")

In [38]:
from IPython.display import Markdown, display
def display_answer(answer):
    display(Markdown("*Query:*\n" + answer["user_query"]))
    display(Markdown("*Response:*\n" + answer["response"]))
    for i, context in enumerate(answer["context"]):
        display(Markdown(f"*Context {i+1}:*\n" + context[0]))

display_answer(answer)

*Query:*
What are the main goals NRC is trying to accomplish through use of AI?

*Response:*
The main goals the NRC is trying to accomplish through the use of AI are outlined in their AI Strategic Plan, which includes five strategic goals:

1. **Ensure NRC readiness for regulatory decision-making**: Prepare the organization to effectively use AI in decision-making processes.
2. **Establish an organizational framework to review AI applications**: Create a structure that encompasses all aspects of the NRC for evaluating AI applications.
3. **Strengthen and expand AI partnerships**: Develop partnerships with industry and relevant stakeholders to ensure safe and secure use of AI.
4. **Cultivate an AI-proficient workforce**: Invest in training and development to equip NRC staff and contractors with the necessary skills to evaluate AI in regulated activities.
5. **Pursue use cases to build an AI foundation across the NRC**: Identify and implement use cases that will help establish a foundational understanding and capability for AI applications within the NRC.

These goals aim to support the NRC’s mission while ensuring safety, security, and effectiveness in evaluating AI in NRC-regulated activities.

*Context 1:*
 to contribute to AI research and development activities.   
The NRC will cultivate the talent of its existing highly skilled workforce by investing in 
comprehensive training for NRC staff and managers working on use cases (AI Strategic 
Goal  5). The NRC AI training program will use a tiered approach, providing training 
ranging from basic to advanced concepts, applications, and AI tools tailored to the needs of the agency and staff development objectives.  
The goal is to adopt the appropriate training programs and tools to develop the 
requisite skills in the NRC workforce. A successful outcome of this goal is to ensure appropriate qualifications, training, expertise, and access to tools exist for the workforce to review and evaluate AI usage in NRC-regulated activities effectively, efficiently and in a timely manner.
4.5  Str ategic Goal 5: Pursue Use Cases to Build an AI Foundation 
Across the NRC
AI technologies may pose novel challenges for the NRC regulatory framework. As the 

*Context 2:*
essfully supporting technical readiness for regulatory decision-making activities desired in AI Strategic Goal 1. The establishment of the organizational framework (AI Strategic Goal 2) ensures all aspects of the NRC are represented in the preparations for reviewing AI in NRC-regulated activities. Strong partnerships are essential to ensuring the safe and secure use of AI in the nuclear industry. As such, the NRC is committed to engaging the industry and relevant stakeholders to maintain awareness of industry efforts (AI Strategic Goal 3) and prepare for regulatory reviews. The NRC will also engage in workforce development and acquisition to ensure that the NRC staff and contractors have the critical skills required (AI Strategic Goal 4) to evaluate the use of AI in NRC-regulated activities. The NRC recognizes the establishment of a foundation in data science as a fundamental requirement for evaluating AI applications. Therefore, the NRC will build the necessary AI foundation to pursue

*Context 3:*
r the NRC to continue to improve its skills and capabilities to review and evaluate the application of AI to NRC-regulated activities, maintain awareness of technological innovations, and ensure the safe and secure use of AI in NRC-regulated activities. The AI Strategic Plan includes five goals: (1) ensure NRC readiness for regulatory decision-making, (2) establish an organizational framework to review AI applications, (3) strengthen and expand AI partnerships, (4) cultivate an AI-proficient workforce, and (5) pursue use cases to build an AI foundation across the NRC. 
The AI Strategic Plan supports the NRC’s mission, broadly aligns with the agency’s 
Principles of Good Regulation, and is tied to multiple NRC FY 2022–2026 Strategic Plan 
safety, security, and openness strategies   [1]  . The o verall goal of the AI Strategic Plan 
is to ensure the staff’s readiness to effectively and efficiently review and evaluate the use of AI in NRC-regulated activities. Any future guidance or rulem

*Context 4:*
ments for new, more detailed models. Additionally, 
the NRC will leverage lessons learned from previous new technology applications in NRC-regulated activities to inform the development of the AI framework. Lastly, additional options for long-range changes for AI regulatory reviews and oversight that might require rulemaking will also be considered. 
The NRC will undertake research to develop an AI framework to determine the 
approach to assess technical areas such as, but not limited to, topics shown in Table 2. The NRC will also work with agency stakeholders and the international regulatory community to determine the currently available AI standards and identify the technical areas where gaps may exist. In addition, the NRC will participate with standards development organizations and work with Federal agencies and the international regulatory community (AI Strategic Goal 3) to offer critical expertise and perspectives to inform the drafting and revision of AI standards and guidance 

In [39]:
answer2 = retrieval_augmented_qa_pipeline.run_pipeline("What challenges does NRC foresee?")
display_answer(answer2)

*Query:*
What challenges does NRC foresee?

*Response:*
The NRC foresees challenges associated with safely and securely deploying, overseeing, and evaluating AI technologies. These challenges may be informed by the experiences and lessons learned from other federal agencies and industry sectors that have more experience with the assessment and implementation of AI. Additionally, there may be challenges in getting AI applications through the AI framework, which could be identified through pilot studies and proofs of concept that rely on industry feedback and engagement.

*Context 1:*
y on industry feedback and engagement, may help in identifying challenges associated with getting the AI applications through the AI framework. 
Lastly, the NRC will investigate improving staff access to software-based AI tools as 
part of the AI ecosystem which may be required to review and evaluate AI applications in NRC-regulated activities. Additionally, providing access to training and development with respect to the tools under the AI ecosystem facilitates staff engagement in training exercises that may mimic future regulatory reviews using such tools. This will allow staff to develop expertise and identify and address potential gaps in future regulatory reviews.  
For this goal, a successful outcome is one in which the NRC staff possesses an 
ecosystem that supports AI analysis, integration of emerging AI tools, and hands-on talent development for reviewing AI applications from the nuclear industry.  McGuire Nuclear Station, Units 1 and 2CONCLUSION
The NRC remains committed to e

*Context 2:*
sting and new memoranda of understanding, public meetings, and workshops. For international AI activities, the NRC will continue to engage with international counterparts and multilateral organizations to collaborate in sharing information on the use of AI in NRC-regulated activities, conduct cooperative research, and influence the development of international standards and guidance. 
The NRC is aware that other Federal agencies and industry sectors are faced with 
the potential challenges of safely and securely deploying, overseeing, and evaluating AI technologies. In some cases, other Government agencies have more experience with assessment and implementation of AI. Their experience and lessons learned provide the NRC with a unique opportunity to engage in intergovernmental information sharing, collaboration, and potential technology transfer from those agencies. The NRC will continue to build partnerships with other Government agencies to facilitate the exchange of ideas, practices,

*Context 3:*
 assessment and integration of emerging AI tools, and hands-on talent development for reviewing the use of AI in NRC-regulated activities. To better understand how AI algorithms, models, and claims are validated and tested, the NRC needs to undertake research to develop use cases with data from various sources and in multiple forms. These use cases will help the staff gain AI expertise that could be used in performing regulatory reviews or assessments for a wide range of potential AI 
4-6
4-7applications. In addition, the NRC is planning to investigate engaging collaboratively 
with the nuclear industry to pursue potential pilot studies and proofs of concept to serve as a foundation for reviewing the use of AI in NRC-regulated activities. These pilots and proofs of concept, which would rely on industry feedback and engagement, may help in identifying challenges associated with getting the AI applications through the AI framework. 
Lastly, the NRC will investigate improving staff access

*Context 4:*
ments for new, more detailed models. Additionally, 
the NRC will leverage lessons learned from previous new technology applications in NRC-regulated activities to inform the development of the AI framework. Lastly, additional options for long-range changes for AI regulatory reviews and oversight that might require rulemaking will also be considered. 
The NRC will undertake research to develop an AI framework to determine the 
approach to assess technical areas such as, but not limited to, topics shown in Table 2. The NRC will also work with agency stakeholders and the international regulatory community to determine the currently available AI standards and identify the technical areas where gaps may exist. In addition, the NRC will participate with standards development organizations and work with Federal agencies and the international regulatory community (AI Strategic Goal 3) to offer critical expertise and perspectives to inform the drafting and revision of AI standards and guidance 

### 🏗️ Activity #1:

Enhance your RAG application in some way! 

Suggestions are: 

- Allow it to work with PDF files
- Implement a new distance metric
- Add metadata support to the vector database

While these are suggestions, you should feel free to make whatever augmentations you desire! 

> NOTE: These additions might require you to work within the `aimakerspace` library - that's expected!

### 🏗️ Summary #1:

1. Added the ability to process PDFs.
1. Added a GUI file explorer to populate the path.
1. Added quality-of-life improvements: read OPEN_AI token from `.env` and pretty-print answers.
